In [ ]:
'''
python version 3.10.12
'''

In [4]:
'''
Make sure to confirm the full path to the requirements.txt file. 
'''
! pip install -r requirements.txt
'''
Please restart this file after executing this cell !!!
'''

In [1]:
import argparse
import sys
import os
import data_utils_LA
import numpy as np
from torch import Tensor
from torch.utils.data import DataLoader
from torchvision import transforms
import yaml
import torch
from torch import nn
from model import RawNet
# from tensorboardX import SummaryWriter
import time
from tqdm import tqdm


In [2]:



def keras_lr_decay(step, decay = 0.0001):
	return 1./(1.+decay*step)

def pad(x, max_len=64600):
    
    x_len = x.shape[0]
    if x_len >= max_len:
        return x[:max_len]
    # need to pad
    num_repeats = int(max_len / x_len)+1
    padded_x = np.tile(x, (1, num_repeats))[:, :max_len][0]
    
    return padded_x 

def init_weights(m):
    #print(m)
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform(m.weight)
        m.bias.data.fill_(0.0001)
    elif isinstance(m, nn.BatchNorm1d):
        pass
    else:
        if hasattr(m, 'weight'):
            torch.nn.init.kaiming_normal_(m.weight, a=0.01)
        else:		
            print('no weight',m)


def evaluate_accuracy(data_loader, model, device):
    num_correct = 0.0
    num_total = 0.0
    model.eval()
    for batch_x, batch_y, batch_meta in data_loader:
        
        batch_size = batch_x.size(0)
        num_total += batch_size
        batch_x = batch_x.to(device)
        batch_y = batch_y.view(-1).type(torch.int64).to(device)
        batch_out = model(batch_x,batch_y)
        _, batch_pred = batch_out.max(dim=1)
        num_correct += (batch_pred == batch_y).sum(dim=0).item()
    return 100 * (num_correct / num_total)


def produce_evaluation_file(dataset, model, device, save_path):

    data_loader = DataLoader(dataset, batch_size=24, shuffle=False)
    num_correct = 0.0
    num_total = 0.0
    model.eval()
    true_y = []
    fname_list = []
    key_list = []
    sys_id_list = []
    key_list = []
    score_list = []
    for batch_x, batch_y, batch_meta in tqdm(data_loader):
        batch_size = batch_x.size(0)
        num_total += batch_size
        batch_x = batch_x.to(device)
        batch_y = batch_y.view(-1).type(torch.int64).to(device)
        batch_out = model(batch_x,batch_y,is_test=True)
        batch_score = (batch_out[:, 1]
                       ).data.cpu().numpy().ravel()

        # add outputs
        fname_list.extend(list(batch_meta[1]))
        key_list.extend(
          ['bonafide' if key == 1 else 'spoof' for key in list(batch_meta[4])])
        sys_id_list.extend(list(batch_meta[3]))
        score_list.extend(batch_score.tolist())
        
    with open(save_path, 'w') as fh:
        for f, s, k, cm in zip(fname_list, sys_id_list, key_list, score_list):
            # if dataset.is_eval:
            fh.write('{} {} {} {}\n'.format(f, s, k, cm))
            # else:
            #     fh.write('{} {}\n'.format(f, cm))
    print('Result saved to {}'.format(save_path))


def train_epoch(data_loader, model, lr,optim, device):
    running_loss = 0
    num_correct = 0.0
    num_total = 0.0
    ii = 0
    model.train()
    weight = torch.FloatTensor([1.0, 9.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight)
    
    for batch_x, batch_y, batch_meta in data_loader:
       
        batch_size = batch_x.size(0)
        num_total += batch_size
        ii += 1
        batch_x = batch_x.to(device)
        batch_y = batch_y.view(-1).type(torch.int64).to(device)
        batch_out = model(batch_x,batch_y)
        batch_loss = criterion(batch_out, batch_y)
        _, batch_pred = batch_out.max(dim=1)
        num_correct += (batch_pred == batch_y).sum(dim=0).item()
        running_loss += (batch_loss.item() * batch_size)
        if ii % 10 == 0:
            sys.stdout.write('\r \t {:.2f}'.format(
                (num_correct/num_total)*100))
        optim.zero_grad()
        batch_loss.backward()
        optim.step()
       
    running_loss /= num_total
    train_accuracy = (num_correct/num_total)*100
    return running_loss, train_accuracy






In [3]:
if __name__ == '__main__':
    parser = argparse.ArgumentParser('ASVSpoof2019  model')
  

    model_path='change this to your RawNet2 model path '# download here [https://huggingface.co/VoiceWukong/VoiceWukong/resolve/main/rawnet2.pth?download=true]
    database_path='/home/ydoit/AIGC/Dataset'

    en_eval_output='change this to the path to save en_eval_scores.txt'
    zh_eval_output='change this to the path to save zh_eval_scores.txt'
    batch_size=24
    num_epochs=100
    lr=0.0001
    weight_decay=0.0001
    comment='ASVSpoof2019-LA'
    track='logical'
    features='Raw_audio'
    eval_part=0
    loss='CCE'
    

    dir_yaml = os.path.splitext('model_config_RawNet2')[0] + '.yaml'

    with open(dir_yaml, 'r') as f_yaml:
            parser1 = yaml.load(f_yaml,Loader=yaml.FullLoader)

    
    np.random.seed(parser1['seed'])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



    
    is_logical = (track == 'logical')
    
    
    transforms = transforms.Compose([
        
        lambda x: pad(x),
        lambda x: Tensor(x)
        
    ])

    # GPU device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'  
    # print('yandoit'+device)# cuda-0
    
    

    
   
    
    
    
    if bool(parser1['mg']):
            model_1gpu = RawNet(parser1['model'], device)
            nb_params = sum([param.view(-1).size()[0] for param in model_1gpu.parameters()])
            model =(model_1gpu).to(device)
    else:
        model = RawNet(parser1['model'], device).to(device)
        nb_params = sum([param.view(-1).size()[0] for param in model.parameters()])
    

    # Adam optimizer
    print('model gen over')
    optimizer = torch.optim.Adam(model.parameters(), lr=lr,weight_decay=weight_decay)
    

    if model_path:
        model.load_state_dict(torch.load(model_path,map_location=device))
        print('Model loaded : {}'.format(model_path))

    

    print('start en eval')
    en_eval_list='change this to the eval_list.txt path'
    en_dev_set = data_utils_LA.ASVDataset(database_path=database_path,protocols_path=en_eval_list,is_train=False, is_logical=is_logical,
                                    transform=transforms,
                                    feature_name=features, is_eval=False, eval_part=eval_part)
    en_dev_loader = DataLoader(en_dev_set, batch_size=batch_size, shuffle=True)
    produce_evaluation_file(en_dev_set, model, device, en_eval_output)
    
    print('end en eval')
    zh_eval_list='change this to the zh_eval_list.txt path'
    zh_dev_set = data_utils_LA.ASVDataset(database_path=database_path,protocols_path=zh_eval_list,is_train=False, is_logical=is_logical,
                                    transform=transforms,
                                    feature_name=features, is_eval=False, eval_part=eval_part)
    zh_dev_loader = DataLoader(zh_dev_set, batch_size=batch_size, shuffle=True)
    
    print('start zh eval')
    
    produce_evaluation_file(zh_dev_set, model, device, zh_eval_output)
    
    print('end zh eval')





model gen over


/tmp/ipykernel_759927/1683333885.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path,map_location=device))


FileNotFoundError: [Errno 2] No such file or directory: 'change this to your RawNet2 model path '